In [4]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Match trader_4 Osmium capacity
POSITION_LIMIT = 80

def load_osmium_series(
    *csv_paths: str,
) -> pd.DataFrame:
    """Concatenate Round-2 price CSVs and return sorted ASH_COATED_OSMIUM rows."""
    frames = [pd.read_csv(p) for p in csv_paths]
    df = pd.concat(frames, ignore_index=True)
    df = df[df["product"] == "ASH_COATED_OSMIUM"].copy()
    return df.sort_values(["day", "timestamp"]).reset_index(drop=True)


def run_backtest_on_term(
    order_book: pd.DataFrame,
    forecast_horizon: int,
    *,
    position_size: int = POSITION_LIMIT,
    step_mode: str = "every_row",
    require_same_day: bool = False,
    verbose: bool = True,
):
    """
    Oracle backtest: at OsMIUM row i we assume we know the full BBO at row i + h.

    Strategy (clairvoyant BBO — same economic assumptions as trader_4 logs: prints are at the touch)
      - Long: buy now at ask_i, unwind at bid_{i+h}  → edge/lot = bid_fut - ask_now
      - Short: sell now at bid_i, cover at ask_fut  → edge/lot = bid_now - ask_fut
      Pick the better positive edge; if both ≤ 0, stay flat.

    Parameters
    ----------
    forecast_horizon : int
        Row offset h in the OsMIUM-only frame (not raw CSV row index).
    step_mode :
        - ``every_row``: one independent scalp per row (mean edge is comparable across h;
          summed PnL assumes unlimited capital — use for ranking horizons).
        - ``non_overlapping``: only i, i+h, i+2h, … so each trade spans h rows without overlap.
    require_same_day :
        If True, skip pairs where ``day`` changes between i and i+h.
    """
    if forecast_horizon < 1:
        raise ValueError("forecast_horizon must be >= 1")
    if step_mode not in ("every_row", "non_overlapping"):
        raise ValueError("step_mode must be 'every_row' or 'non_overlapping'")

    bid = order_book["bid_price_1"].to_numpy(dtype=float)
    ask = order_book["ask_price_1"].to_numpy(dtype=float)
    days = order_book["day"].to_numpy() if require_same_day else None
    n = len(order_book)

    idxs = range(0, n - forecast_horizon)
    if step_mode == "non_overlapping":
        idxs = range(0, n - forecast_horizon, forecast_horizon)

    long_edges: list[float] = []
    short_edges: list[float] = []
    chosen: list[float] = []

    iter_idx = list(idxs)
    if verbose:
        iter_idx = tqdm(iter_idx, desc=f"h={forecast_horizon}", leave=False)
    for i in iter_idx:
        j = i + forecast_horizon
        if require_same_day and days is not None and days[i] != days[j]:
            continue

        bn, an = bid[i], ask[i]
        bf, af = bid[j], ask[j]
        if any(np.isnan(x) for x in (bn, an, bf, af)):
            continue

        le = bf - an  # long round-trip (lift ask now, sell bid later)
        se = bn - af  # short round-trip (hit bid now, cover ask later)
        long_edges.append(le)
        short_edges.append(se)
        if le <= 0 and se <= 0:
            chosen.append(0.0)
        else:
            chosen.append(max(le, se))

    long_edges_a = np.array(long_edges, dtype=float)
    short_edges_a = np.array(short_edges, dtype=float)
    chosen_a = np.array(chosen, dtype=float)

    n_dec = len(chosen_a)
    mean_best_edge = float(chosen_a.mean()) if n_dec else float("nan")
    hit_rate = float((chosen_a > 0).mean()) if n_dec else float("nan")
    total_pnl = float(chosen_a.sum() * position_size)

    return {
        "horizon": forecast_horizon,
        "n_decisions": n_dec,
        "mean_best_edge_per_lot": mean_best_edge,
        "hit_rate_positive_edge": hit_rate,
        "total_pnl_scaled": total_pnl,
        "mean_long_edge": float(long_edges_a.mean()) if n_dec else float("nan"),
        "mean_short_edge": float(short_edges_a.mean()) if n_dec else float("nan"),
        "step_mode": step_mode,
    }


def sweep_horizons(
    order_book: pd.DataFrame,
    horizons: range | list[int],
    **kwargs,
) -> pd.DataFrame:
    kw = {**kwargs, "verbose": False}
    rows = [run_backtest_on_term(order_book, h, **kw) for h in horizons]
    return pd.DataFrame(rows)




In [5]:
from pathlib import Path

# Notebook cwd should be ROUND_2 (same as original relative CSV paths).
BASE = Path(".")
os_df = load_osmium_series(
    str(BASE / "prices_round_2_day_-1.csv"),
    str(BASE / "prices_round_2_day_0.csv"),
    str(BASE / "prices_round_2_day_1.csv"),
)
print(len(os_df), "ASH_COATED_OSMIUM rows (days -1, 0, 1)")
os_df.head()

30000 ASH_COATED_OSMIUM rows (days -1, 0, 1)


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-1,0,ASH_COATED_OSMIUM,9982.0,21.0,NaN,NaN,NaN,NaN,10000.0,13.0,10003.0,21.0,NaN,NaN,9991.0,0.0
1,-1,100,ASH_COATED_OSMIUM,9984.0,15.0,NaN,NaN,NaN,NaN,10000.0,15.0,10003.0,26.0,NaN,NaN,9992.0,0.0
2,-1,200,ASH_COATED_OSMIUM,9985.0,15.0,9982.0,30.0,NaN,NaN,10001.0,15.0,NaN,NaN,NaN,NaN,9993.0,0.0
3,-1,300,ASH_COATED_OSMIUM,9982.0,27.0,NaN,NaN,NaN,NaN,10003.0,27.0,NaN,NaN,NaN,NaN,9992.5,0.0
4,-1,400,ASH_COATED_OSMIUM,9995.0,8.0,9985.0,12.0,9983.0,25.0,10001.0,12.0,10004.0,25.0,NaN,NaN,9998.0,0.0


In [13]:
# Horizon sweep: h = OsMIUM row offset (100 ms ticks within merged series).
# Compare horizons by mean_best_edge_per_lot (every_row) or total_pnl_scaled (non_overlapping).

H_MAX = 100
STEP = 1
hs = range(1, H_MAX + 1, STEP)

summary_every = sweep_horizons(os_df, hs, step_mode="every_row")
summary_no = sweep_horizons(os_df, hs, step_mode="non_overlapping")

best_mean = summary_every.sort_values("mean_best_edge_per_lot", ascending=False).head(12)
best_cash = summary_no.sort_values("total_pnl_scaled", ascending=False).head(12)
display(best_mean)
display(best_cash)

h_star = int(summary_every.loc[summary_every["mean_best_edge_per_lot"].idxmax(), "horizon"])
print("horizon with best mean clairvoyant edge (every_row):", h_star)
run_backtest_on_term(os_df, h_star, step_mode="every_row", verbose=True)

,horizon,n_decisions,mean_best_edge_per_lot,hit_rate_positive_edge,total_pnl_scaled,mean_long_edge,mean_short_edge,step_mode
99,100,25495,0.006080,0.002000,12400.0,-16.243342,-16.228515,every_row
89,90,25507,0.005685,0.002078,11600.0,-16.246717,-16.221155,every_row
91,92,25524,0.005563,0.002155,11360.0,-16.241537,-16.221086,every_row
78,79,25513,0.004939,0.001764,10080.0,-16.242817,-16.227061,every_row
51,52,25509,0.004783,0.001215,9760.0,-16.235917,-16.229644,every_row
88,89,25520,0.004741,0.001724,9680.0,-16.241458,-16.220846,every_row
83,84,25522,0.004623,0.001606,9440.0,-16.247865,-16.225805,every_row
76,77,25511,0.004586,0.001568,9360.0,-16.237819,-16.231822,every_row
95,96,25519,0.004271,0.001724,8720.0,-16.248090,-16.218543,every_row
93,94,25516,0.004233,0.001764,8640.0,-16.244043,-16.214414,every_row


,horizon,n_decisions,mean_best_edge_per_lot,hit_rate_positive_edge,total_pnl_scaled,mean_long_edge,mean_short_edge,step_mode
0,1,25599,0.000938,0.000273,1920.0,-16.224618,-16.241533,non_overlapping
1,2,12763,0.001567,0.000470,1600.0,-16.243516,-16.237875,non_overlapping
16,17,1507,0.009290,0.002654,1120.0,-16.244194,-16.239549,non_overlapping
12,13,1969,0.007110,0.003047,1120.0,-16.160995,-16.114779,non_overlapping
3,4,6396,0.001563,0.000469,800.0,-16.209193,-16.200125,non_overlapping
9,10,2570,0.003502,0.001556,720.0,-16.259533,-16.236576,non_overlapping
21,22,1159,0.007765,0.002588,720.0,-16.257118,-16.267472,non_overlapping
99,100,248,0.036290,0.004032,720.0,-16.157258,-16.060484,non_overlapping
8,9,2835,0.002822,0.001058,640.0,-16.217989,-16.196473,non_overlapping
42,43,606,0.011551,0.003300,560.0,-16.179868,-16.207921,non_overlapping


horizon with best mean clairvoyant edge (every_row): 100


{'horizon': 100,
 'n_decisions': 25495,
 'mean_best_edge_per_lot': 0.006079623455579526,
 'hit_rate_positive_edge': 0.0020003922337713275,
 'total_pnl_scaled': 12400.0,
 'mean_long_edge': -16.243341831731712,
 'mean_short_edge': -16.228515395175524,
 'step_mode': 'every_row'}

In [ ]:
# Optional: only pair rows within the same `day` column (no oracle across day -1/0/1 boundaries).
run_backtest_on_term(os_df, 20, require_same_day=True, verbose=False)